## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from modelbase.ode import Model, Simulator, mca
import pandas as pd
import seaborn as sns
import math
import pickle
from model import (
    get_enzyme_costs,
    get_model,
    get_enzyme_concentrations,
    get_enzyme_parameters,
    get_proteomics_results,
    calculate_min_kcats,
)
from modelbase.ode.simulators import _Simulate
from parallel import parallelise
import itertools
from pathlib import Path
from sklearn.pipeline import Pipeline
from typing import Literal
from tqdm import tqdm
from plot import savefig
from functools import partial
from typing import TypeVar


T = TypeVar("T")


def unwrap(x: T | None) -> T:
    if x is None:
        raise ValueError("Unexpected None")
    return x


CELL_VOLUME: float = 2e-15
PROTEIN_CONTENT: float = 0.445
MODE = "blast"
CHLA = (
    7.5 / 893.509 / (105482164.18012363 * 1000)
)  # mg/OD/L -> mmol/OD/L / Cells/LLOD --> mmol/cell
CAROT = 2 / 581.5565 / (105482164.18012363 * 1000)
SQUALENE = 0.5 / 410.73 / (105482164.18012363 * 1000)
DOUBLING_TIME: float = 300.0

# Load data
DATA = Path("data")

with open(DATA / "rate_regression_model_biomass.pickle", "rb") as f:
    BIOMASS_REGRESSION_MODEL = pickle.load(f)

OCP_ABSORPTION = pd.read_csv(DATA / "OCP_absorption.csv", index_col=0)
PS_COMP = pd.read_csv(DATA / "photosystem_composition.csv", index_col=0)
MOLAR_MASSES = pd.read_csv(DATA / "molar_masses.csv", index_col=0).squeeze(1)

PIGMENTS_FUENTE = pd.read_csv(
    DATA / "pigment_concentrations.csv",
    index_col=0,
).squeeze(1)

ABS_COEF = pd.concat(
    {
        pig: pd.read_csv(DATA / f"per_pigment/{pig}.csv", index_col=0)
        for pig in PIGMENTS_FUENTE.index
    },
    axis=1,
)

ENZYME_COSTS = get_enzyme_costs(
    kugler_model_csv=DATA / "Amino_acid_costs_Kuglermodel.csv",
    aa_costs_csv=DATA / "Amino_acid_costs_arnold.csv",
)

ABUNDANCES, DETECTION_LIMIT = get_proteomics_results(
    proteomics_diff_csv=DATA / "Proteomics_DiffAnalysis_2WayAnova_.csv",
    proteomics_poi_csv=DATA / "Proteomics_Proteins_of_interest.csv",
    proteomics_concentration_csv=DATA / "Proteomics_samples_proteinconcentration.csv",
    proteomics_dry_weight_csv=DATA / "Proteomics_Dry_weights.csv",
    proteomics_dilutions_csv=DATA / "Proteomics_samples_dilutions.csv",
    proteomics_od_wt_csv=DATA / "20240328_Spectrum_OD_WT_proteomics.csv",
    proteomics_od_sqs_csv=DATA / "20240329_Spectrum_OD_sqs_proteomics.csv",
    proteomics_od_dcas9_csv=DATA / "20240329_Spectrum_OD_dCas9_proteomics.csv",
    cell_volume=CELL_VOLUME,
    protein_content=PROTEIN_CONTENT,
)

ENZYME_CONCENTRATIONS_UMOL: dict[str, float] = get_enzyme_concentrations(
    equilibrium_constants_csv=DATA / "Equilibrium_constants.csv",
    abundances=ABUNDANCES,
    detection_limit=DETECTION_LIMIT,
)

ENZYME_PARAMETERS: dict[str, float] = get_enzyme_parameters(
    mode=MODE,
    blasted_parameters_csv=DATA / "enzyme_parameters_blasted.csv",
    predicted_kms_csv=DATA / "predicted_Kms.csv",
    equilibrium_constants_csv=DATA / "Equilibrium_constants.csv",
)

MIN_KCAT_VALUES: dict[str, float] = calculate_min_kcats(
    enzyme_concentrations_umol=ENZYME_CONCENTRATIONS_UMOL,
    cell_volume=CELL_VOLUME,
    doubling_time=DOUBLING_TIME,
)


LIGHTS = pd.DataFrame(
    {
        path.stem: pd.read_csv(path, index_col=0).squeeze(1)
        for path in sorted((DATA / "lights").glob("*.csv"))
    }
)
EXP_GROWTH_RATE_BY_DENSITY = pd.read_csv(
    DATA / "growth_rates_by_density.csv",
    index_col=0,
)["growth_rate"]

In [ ]:
def get_parameterised_model(
    strain: Literal["wt", "dshc"],
    growth_mode: Literal["steady", "dynamic"],
    kcat_values: dict[str, float] = MIN_KCAT_VALUES,
    cell_volume: float = CELL_VOLUME,
    enzyme_costs: pd.DataFrame = ENZYME_COSTS,
    enzyme_concentrations_umol: dict[str, float] = ENZYME_CONCENTRATIONS_UMOL,
    enzyme_parameters: dict[str, float] = ENZYME_PARAMETERS,
    biomass_regression_model: Pipeline = BIOMASS_REGRESSION_MODEL,
    abs_coef: pd.DataFrame = ABS_COEF,
    lights: pd.DataFrame = LIGHTS,
    ocp_absorption: pd.DataFrame = OCP_ABSORPTION,
    molar_masses: pd.DataFrame = MOLAR_MASSES,
    ps_comp: pd.DataFrame = PS_COMP,
) -> tuple[Model, dict[str, float]]:
    m = get_model(
        growth_mode=growth_mode,
        strain=strain,
        cell_volume=cell_volume,
        enzyme_total_cost_mol=enzyme_costs,
        enzyme_concentrations_umol=enzyme_concentrations_umol,
        enzyme_parameters=enzyme_parameters,
        biomass_regression_model=biomass_regression_model,
        abs_coef=abs_coef,
        lights=lights,
        ocp_absorption=ocp_absorption,
        molar_masses=molar_masses,
        ps_comp=ps_comp,
    )
    m.update_parameters(
        {
            "ShcKcat": 0,
            "Shc_ksynth": 0,
            "light": 100,
            "ATP_out_metabolism": 0.5,
            "3PGA_out_metabolism": 0.5,
            "NADPH_out_metabolism": 0.5,
            "Fdred_out_metabolism": 0.5,
            "out": 0,
            "cell_volume": cell_volume,
            "protein_content": 0.445,
            "DXS_inh_str": 0.001,
            "ThiGDXPKm": 0.01,
        }
    )
    m.update_parameters(kcat_values)

    # Update initial
    starting_conc = {}
    for i in m.compounds:
        starting_conc[i] = 0.01
    starting_conc.update(enzyme_concentrations_umol)
    starting_conc.update(
        {
            "3PGA": 0.1,
            "ATP": 0.1,
            "NADPH": 0.1,
            "Fdred": 0.1,
            "ThiDP": 10,
            "chlorophyll-phytol": CHLA / cell_volume,
            "Cell_density": 1e1,
            "carotenoids": CAROT / cell_volume,
            "SQ": SQUALENE / cell_volume,
        }
    )
    return m, starting_conc


def get_doubling_time(rel_growth_rate: float) -> float:
    return np.log(2) / (rel_growth_rate * 60)


In [ ]:
# 3 µM/min 3PGA at OD=1, growth rate 0.001 == doubling time around 11 h.
Start_OD = 1
Growth_rate = 0.001  # min^-1
print(np.log(2) / 0.001 / 60)
print(
    "g/L/min calculated from growth: ", Start_OD * (Growth_rate) / 3 / 2
)  # OD*Growth_rate min^-1--> OD/min / g/L CDW/OD --> g/L CDW/min * gC/gCDW --> g/L/min
print(
    "g/L/min calculated from influx prediction:",
    14 * 2e-15 * 1e11 * 186.06 * (36 / 186.06) * 1e-3,
)  # mmol/L/min 3PGA * L/cell--> mmol/cell/min *cells/L --> mmol/L/min * mg/mmol -> mg 3PGA /L/min --> mg C/L/min --> g/L/min
# 0.06 fg/cell s^-1
print(
    "fg/cell/s to compare to paper:", 2.4 * 2e-15 * 186.061 * 1e9 / 60
)  # mmol/L/min --> mmol/cell/min --> mg/cell/min --> fg/cell/min -_> fg/cell/s

print(
    "g/L/min calculated from fg/cell/s value in Paper:", 0.06 * 1e11 * 60 / 1e15
)  # fg/cell/s * cells/L --> fg/L/s --> fg/L/min --> g/L/min

## Fig XX: compare min kcat with literature

In [ ]:
with plt.rc_context(
    {
        "lines.linewidth": 3,
        "font.size": 12.0,
        "axes.grid": True,
    }
):
    fig, ax = plt.subplots(figsize=(5, 3.5))
    ax.set_axisbelow(b=True)

    kcat_comp = pd.DataFrame.from_dict(MIN_KCAT_VALUES, "index")
    for key, value in ENZYME_PARAMETERS.items():
        if key in kcat_comp.index:
            kcat_comp.at[key, "A"] = value
    kcat_comp.plot(kind="bar", ax=ax)

    ax.legend([r"Calculated min $\mathrm{k_{cat}}$", r"Literature $\mathrm{k_{cat}}$"])

## Fig XX: growth rate over time

In [ ]:
# Get steady-state
ms, y0 = get_parameterised_model(
    strain="dshc",
    growth_mode="steady",
)
ms.update_parameters(
    {
        "light": 80,
        "DXS_inh_str": 0,  # FIXME: why is this changed?
        "sample_depth_m": 0.0135,  # FIXME: why is this changed?
    }
)

y0_ss = unwrap(
    Simulator(ms)
    .initialise(y0 | {"Cell_density": EXP_GROWTH_RATE_BY_DENSITY.index[24] / 1e6})
    .simulate_to_steady_state_and()
    .get_new_y0()
)

# Run dynamic simulation
md, _ = get_parameterised_model(
    strain="dshc",
    growth_mode="dynamic",
)
md.update_parameters(
    {
        "light": 80,
        "sample_depth_m": 0.0135,  # FIXME: why is this changed?
    }
)

# FIXME: explain why this number
res = unwrap(
    Simulator(md)
    .initialise(y0_ss | {"Cell_density": 54562873.0217531 / 1e6})
    .simulate_and(96 * 60)
    .get_full_results_df()
)

In [ ]:
with plt.rc_context(
    {
        "lines.linewidth": 3,
        "font.size": 12.0,
    }
):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3), layout="constrained")

    ax1.plot(res["Growth_rate"] * 60)
    ax1.scatter(
        x=np.linspace(0, 96 * 60, len(EXP_GROWTH_RATE_BY_DENSITY)),
        y=EXP_GROWTH_RATE_BY_DENSITY,
        c="C1",
    )
    ax1.set_ylabel("Growth rate $[h^{-1}]$")
    ax1.set_xlabel("Time [h]")
    ax1.set_xticks(np.arange(0, 97 * 60, 24 * 60))
    ax1.set_xticklabels(np.arange(0, 97, 24))
    ax1.grid()

    ax2.plot(
        res["Cell_density"] * 1e6,
        label="Simulated values",
    )
    ax2.plot(
        np.linspace(0, 96 * 60, len(EXP_GROWTH_RATE_BY_DENSITY[::2])),
        EXP_GROWTH_RATE_BY_DENSITY[::2].index,
        label="Experimental values",
    )
    ax2.set_yscale("log")
    ax2.set_ylabel(r"Cell density $[Cells\ mL^{-1}]$")
    ax2.set_xlabel("Time [h]")
    ax2.set_xticks(np.arange(0, 97 * 60, 24 * 60))
    ax2.set_xticklabels(np.arange(0, 97, 24))
    ax2.grid()

    ax2.legend(loc="lower right")
    savefig(fig, "fig1.png")
    plt.show()

## Fig XX: scan ss growth rate by cell density

In [ ]:
def _worker(
    density: float,
    model: Model,
    y0: dict[str, float],
) -> float | None:
    try:
        df = (
            Simulator(model)
            .initialise(y0 | {"Cell_density": density / 1e6})
            .simulate_to_steady_state_and(
                # simulation_kwargs={"atol": 1e-8, "rtol": 1e-8}
            )
            .get_full_results_df()
        )
        if df is not None:
            return df.iloc[-1]["Growth_rate"] * 60
    except:  # noqa: E722
        print(f"Failed for density={density}")
    return None


ms, y0 = get_parameterised_model(
    strain="dshc",
    growth_mode="steady",
)
ms.update_parameters(
    {
        "light": 80,
        "sample_depth_m": 0.02,
    }
)

growth_rate_by_density = pd.Series(
    parallelise(
        partial(_worker, model=ms, y0=y0),
        inputs=list(
            zip(
                EXP_GROWTH_RATE_BY_DENSITY.index[::4],  # FIXME: 2
                EXP_GROWTH_RATE_BY_DENSITY.index[::4],  # FIXME: 2
                strict=True,
            )
        ),
    ),
    dtype=float,
)


In [ ]:
with plt.rc_context(
    {
        "lines.linewidth": 3,
        "font.size": 12.0,
        "axes.grid": True,
    }
):
    fig, ax = plt.subplots(figsize=(4, 3))
    growth_rate_by_density.plot(ax=ax, label="Simulation")
    EXP_GROWTH_RATE_BY_DENSITY.plot(ax=ax, marker="o", label="Experimental")
    ax.set_xscale("log")
    ax.set_ylim(0)
    ax.set_ylabel(r"Growth rate [$h^{-1}$]")
    ax.set_xlabel(r"Cell density [$Cells\ ml^{-1}$]")
    ax.legend()


### Steady state productivities at different cell densities, light intensities and reactor depths

In [ ]:
ms, y0 = get_parameterised_model(
    growth_mode="steady",
    strain="dshc",
)
ms.update_parameters(
    {
        "sample_depth_m": 0.0325,
    }
)

s = Simulator(ms).initialise(y0)
t_ss, y_ss = s.simulate_to_steady_state()
y0_ss = dict(zip(ms.compounds, y_ss[-1]))
for l in np.arange(100, 501, 50):
    conc_collection_densities = []
    flux_collection_densities = []
    for d in np.logspace(1, 3.6, 10):
        s = Simulator(ms)
        y0_ss.update({"Cell_density": d})
        s.initialise(y0_ss)
        ms.update_parameters({"light": l})
        try:
            t_steady, y_ss = s.simulate_to_steady_state()
            print(list(zip(ms.compounds, y_ss[-1])))
            conc_collection_densities.append(s.get_full_results_df())
            flux_collection_densities.append(s.get_fluxes_df())
        except:
            pass
    all_runs[l] = conc_collection_densities

In [ ]:
fig, ax = plt.subplots()
colors = ["blue", "red", "black", "orange", "green", "violet", "pink", "yellow", "teal"]
all_rates = []
for o, l in enumerate(all_runs):
    growth_rates_simulated = []
    densities = []

    for i, v in enumerate(all_runs[l]):
        growth_rates_simulated.append(v["Growth_rate"] * v["Cell_density"] * 60 * 1e6)
        densities.append(v["Cell_density"] * 1e6)
    all_rates.append(growth_rates_simulated)
    ax.plot(densities, growth_rates_simulated, c=colors[o])


ax.set_ylim(0)
ax.set_ylabel("Biomass productivity [$Cells\ h^{-1}$]")
ax.set_xlabel("Cell density [$Cells\ ml^{-1}$]")
ax.legend(np.arange(100, 501, 50))

### Growth rate simulation depending on light intensity

In [ ]:
ms = get_model(
    mode="blast", growth_mode="steady", cell_volume=CELL_VOLUME, strain="dshc"
)
ms.update_parameters(
    {
        "light": 80,
        "ATP_out_metabolism": 0.5,
        "3PGA_out_metabolism": 0.5,
        "NADPH_out_metabolism": 0.5,
        "Fdred_out_metabolism": 0.5,
        "out": 0,
        "cell_volume": CELL_VOLUME,
        "protein_content": 0.445,
        "DXS_inh_str": 0.001,
        "ThiGDXPKm": 0.01,
        "sample_depth_m": 0.02,
    }
)
print(ms.get_parameter("cell_volume"))
# m.update_parameters(new_parameters)

ms.update_parameters(MIN_KCAT_VALUES)
conc_collection_light = {}
flux_collection_light = []
for i in np.arange(50, 400, 50):
    ms.update_parameters({"light": i})
    s = Simulator(ms)
    s.initialise(y0)
    try:
        t_steady, y_ss = s.simulate_to_steady_state()
        conc_collection_light[i] = s.get_full_results_df()
        flux_collection_light[i] = s.get_fluxes_df()
    except:
        pass
    if type(y_ss) == None:
        print(i, "no steady state")

In [ ]:
fig, ax = plt.subplots()
growth_rate = {}
for i in conc_collection_light.keys():
    try:
        growth_rate[i] = conc_collection_light[i]["Growth_rate"] * 60
    except:
        pass
print(list(growth_rate.values()))
ax.scatter(y=list(growth_rate.values()), x=list(growth_rate.keys()))
ax.set_ylim(0)
ax.set_ylabel("Growth rate [$h^{-1}$]")
ax.set_xlabel("Light intensity [$µE\ m^{-2}\ s^{-1}$]")


In [ ]:
ms = get_model(
    mode="blast", growth_mode="steady", cell_volume=CELL_VOLUME, strain="dshc"
)
ms.update_parameters(
    {
        "light": 200,
        "ATP_out_metabolism": 0.5,
        "3PGA_out_metabolism": 0.5,
        "NADPH_out_metabolism": 0.5,
        "Fdred_out_metabolism": 0.5,
        "out": 0,
        "cell_volume": CELL_VOLUME,
        "protein_content": 0.445,
        "DXS_inh_str": 0.001,
        "ThiGDXPKm": 0.01,
        "sample_depth_m": 0.02,
    }
)
print(ms.get_parameter("cell_volume"))

c_flux = []
conc_collection = []
flux_collection = []
s = Simulator(ms)
print(ms.get_fluxes_dict(y0)["ATP production"])
s.initialise(y0)

t_steady, y_ss = s.simulate_to_steady_state()
y0_ss = dict(zip(ms.compounds, y_ss[-1]))

### Growth repression induction time point tests
#### Unmodified dshc strain

In [ ]:
def repress_crte(s: _Simulate) -> _Simulate:
    s.update_parameter("CrtE3Kcat", 0.01)
    return s


def product_after_induction(time_point_of_induction: float) -> float:
    kcat_values = calculate_min_kcats(cell_volume=CELL_VOLUME, doubling_time=300)
    m = get_model(
        mode="blast", growth_mode="dynamic", cell_volume=CELL_VOLUME, strain="dshc"
    )
    m.update_parameters(
        {
            "light": 100,
            "ATP_out_metabolism": 0.5,
            "3PGA_out_metabolism": 0.5,
            "NADPH_out_metabolism": 0.5,
            "Fdred_out_metabolism": 0.5,
            "out": 0,
            "cell_volume": CELL_VOLUME,
            "protein_content": 0.445,
            "DXS_inh_str": 0.001,
        }
    )
    m.update_parameters(kcat_values)
    s = Simulator(m)
    s.initialise(y0_ss)
    # Simulate until the time point of induction
    s.simulate(time_point_of_induction)
    # Repress CrtE and continue the simulation for the remaining time
    s = repress_crte(s)
    s.simulate(fourteen_days)

    if (c := s.get_full_results_df()) is None:
        return 0

    cl = c.iloc[-1]
    return cl


day_in_min = 60 * 24 / 2
fourteen_days = 14 * day_in_min * 2
time_points_of_induction = np.arange(0, fourteen_days, day_in_min, dtype=float)
_res = parallelise(
    product_after_induction,
    list(zip(time_points_of_induction, time_points_of_induction, strict=True)),
)

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(10, 20))
for r in _res:
    ax[0].scatter(
        x=r,
        y=_res[r]["Cell_density"] * 1e9 * _res[r]["SQ"] * 410.73 * CELL_VOLUME,
        c="",
    )  # Conversion from intracellular mM to mg/L squalene
    ax[1].scatter(x=r, y=_res[r]["Cell_density"] * 1e9, c="green")
    ax[2].scatter(x=r, y=_res[r]["SQ"] * 410.73 / 1000, c="blue")
    # plt.scatter(x=r, y=res[r]["Cell_density"]*)
ax[0].set_xticks(np.arange(0, 15 * 60 * 24, 1 * 60 * 24))
ax[0].set_xticklabels(np.arange(0, 15, 1))
ax[1].set_xticks(np.arange(0, 15 * 60 * 24, 1 * 60 * 24))
ax[1].set_xticklabels(np.arange(0, 15, 1))
ax[2].set_xticks(np.arange(0, 15 * 60 * 24, 1 * 60 * 24))
ax[2].set_xticklabels(np.arange(0, 15, 1))
ax[0].set_ylabel("Squalene concentration in culture after 14 d [$mg\ L^{-1}$]")

ax[1].set_ylabel("Cell density after 14 d [$Cells\ L^{-1}$]")
ax[2].set_ylabel("Intracellular squalene concentration after 14 d [$g\ L^{-1}$]")
ax[0].set_xlabel("Timepoint of induction [d]")
ax[1].set_xlabel("Timepoint of induction [d]")
ax[2].set_xlabel("Timepoint of induction [d]")

#### Quadruple overexpression of Sqs, DXS, IspD, IspG

In [ ]:
def product_after_induction_ox(time_point_of_induction: float) -> float:
    poi = pd.read_csv(
        "../../assays/Proteomics/Proteomics_Proteins_of_interest.csv", delimiter=";"
    ).drop([0, 15], axis=0)
    kcat_values = calculate_min_kcats(cell_volume=CELL_VOLUME, doubling_time=300)
    m = get_model(
        mode="blast", growth_mode="dynamic", cell_volume=CELL_VOLUME, strain="dshc"
    )
    m.update_parameters(
        {
            "light": 100,
            "ATP_out_metabolism": 0.5,
            "3PGA_out_metabolism": 0.5,
            "NADPH_out_metabolism": 0.5,
            "Fdred_out_metabolism": 0.5,
            "out": 0,
            "cell_volume": CELL_VOLUME,
            "protein_content": 0.445,
            "DXS_inh_str": 0.001,
        }
    )
    for gene in ["Sqs", "IspD", "IspG", "DXS"]:
        overexpression_amount = 1.331057 * poi[poi["Enzyme"] == gene]["kDa"]
        m.update_parameters(
            {
                gene + "_ksynth": m.get_parameter(gene + "_ksynth")
                + overexpression_amount / (5.5 * 60)
            }
        )
    m.update_parameters(kcat_values)
    s = Simulator(m)
    s.initialise(y0_ss)
    # Simulate until the time point of induction
    s.simulate(time_point_of_induction)
    # Repress CrtE and continue the simulation for the remaining time
    s = repress_crte(s)
    s.simulate(fourteen_days)

    if (c := s.get_full_results_df()) is None:
        return 0

    cl = c.iloc[-1]
    return cl


day_in_min = 60 * 24 / 2
fourteen_days = 14 * day_in_min * 2
time_points_of_induction = np.arange(0, fourteen_days, day_in_min, dtype=float)
res_ox_strain = parallelise(
    product_after_induction_ox,
    list(zip(time_points_of_induction, time_points_of_induction, strict=True)),
)

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(10, 20))
for r in res_ox_strain:
    ax[0].scatter(
        x=r,
        y=res_ox_strain[r]["Cell_density"]
        * 1e9
        * res_ox_strain[r]["SQ"]
        * 410.73
        * CELL_VOLUME,
        c="blue",
    )  # Conversion from intracellular mM to mg/L squalene
    ax[1].scatter(x=r, y=res_ox_strain[r]["Cell_density"] * 1e9, c="green")
    ax[2].scatter(x=r, y=res_ox_strain[r]["SQ"] * 410.73 / 1000, c="blue")
    # plt.scatter(x=r, y=res_ox_strain[r]["Cell_density"]*)
ax[0].set_xticks(np.arange(0, 15 * 60 * 24, 1 * 60 * 24))
ax[0].set_xticklabels(np.arange(0, 15, 1))
ax[1].set_xticks(np.arange(0, 15 * 60 * 24, 1 * 60 * 24))
ax[1].set_xticklabels(np.arange(0, 15, 1))
ax[2].set_xticks(np.arange(0, 15 * 60 * 24, 1 * 60 * 24))
ax[2].set_xticklabels(np.arange(0, 15, 1))
ax[0].set_ylabel("Squalene concentration in culture after 14 d [$mg\ L^{-1}$]")

ax[1].set_ylabel("Cell density after 14 d [$Cells\ L^{-1}$]")
ax[2].set_ylabel("Intracellular squalene concentration after 14 d [$g\ L^{-1}$]")
ax[0].set_xlabel("Timepoint of induction [d]")
ax[1].set_xlabel("Timepoint of induction [d]")
ax[2].set_xlabel("Timepoint of induction [d]")

In [ ]:
ms = get_parameterised_model()

s = Simulator(ms)
s.initialise(y0)
t_ss, y_ss = s.simulate_to_steady_state()
y0_ss = dict(zip(ms.compounds, y_ss[-1]))
y0_ss

In [ ]:
get_doubling_time(s.get_full_results_df()["Growth_rate"])

### Simulation of overexpressions in different combinations, maybe include different strengths?

In [ ]:
def_model = get_parameterised_model()

s = Simulator(def_model)
s.initialise(y0)
t_ss, y_ss = s.simulate_to_steady_state()
y0_ss = dict(zip(def_model.compounds, y_ss[-1]))

CELL_VOLUME = 2e-15
MIN_KCAT_VALUES = calculate_min_kcats(cell_volume=CELL_VOLUME, doubling_time=300)
ms = get_model(mode="blast", growth_mode="dynamic", strain="dshc")
ms.update_parameters(
    {
        "ShcKcat": 0,
        "Shc_ksynth": 0,
        "light": 100,
        "ATP_out_metabolism": 0.5,
        "3PGA_out_metabolism": 0.5,
        "NADPH_out_metabolism": 0.5,
        "Fdred_out_metabolism": 0.5,
        "out": 0,  # .0003137085008420509,
        # "ATP_tot":430.343, "NADP_tot":26.805, "3PGA_max":1000, "Fd_tot":3.597,
        "cell_volume": CELL_VOLUME,
        "protein_content": 0.445,
        "DXS_inh_str": 0.001,
    }
)
ms.update_parameters(MIN_KCAT_VALUES)
poi = pd.read_csv(
    "../../assays/Proteomics/Proteomics_Proteins_of_interest.csv", delimiter=";"
).drop([0, 15], axis=0)

c_flux = []
conc_collection_ox = []
flux_collection_ox = []

genes = ["Sqs", "IspG", "DXS", "IspD", "Idi"]

# Generate all possible combinations of the genes
for r in range(1, len(genes) + 1):
    for combination in itertools.combinations(genes, r):
        ms = get_model(mode="blast", growth_mode="steady", strain="dshc")
        for gene in combination:
            overexpression_amount = 1.331057 * poi[poi["Enzyme"] == gene]["kDa"]
            ms.update_parameters(
                {
                    gene + "_ksynth": def_model.get_parameter(gene + "_ksynth")
                    + overexpression_amount
                }
            )
        ms.update_parameters(
            {
                "light": 100,
                "ATP_out_metabolism": 0.5,
                "3PGA_out_metabolism": 0.5,
                "NADPH_out_metabolism": 0.5,
                "Fdred_out_metabolism": 0.5,
                "out": 0,  # .0003137085008420509,
                # "ATP_tot":430.143, "NADP_tot":26.805, "3PGA_max":1000, "Fd_tot":3.597,
                "cell_volume": CELL_VOLUME,
                "protein_content": 0.445,
                "DXS_inh_str": 0.001,
            }
        )
        ms.update_parameters(MIN_KCAT_VALUES)
        s = Simulator(ms)
        y_ss_modified = dict((zip(ms.compounds, y_ss[-1])))
        y_ss_modified.update({"Cell_density": 1e2})
        s.initialise(y0_ss)
        t_steady, y_ss = s.simulate_to_steady_state()
        conc_collection_ox.append(s.get_full_results_df())
        flux_collection_ox.append(s.get_fluxes_df())


In [ ]:
# Assuming conc_collection is a list of DataFrames
# and each DataFrame has a column named "SQ"
fig, ax = plt.subplots(figsize=(10, 10))
# Extract the "SQ" values from each DataFrame in conc_collection
print([df for df in flux_collection_ox])

sq_values = [
    df["Sqs"].iloc[-1] / df["Biomass"].iloc[-1] * 410.73 * 1e3 * 1e9 * CELL_VOLUME
    for df in flux_collection_ox
]

# Generate the labels for the legend based on the gene combinations
genes = ["Sqs", "IspG", "DXS", "IspD", "Idi"]
labels = []
for r in range(1, len(genes) + 1):
    for combination in itertools.combinations(genes, r):
        labels.append(", ".join(combination))

# Create a bar plot
x = np.arange(len(sq_values))
plt.bar(x, sq_values)

# Add labels and legend
ax.set_xlabel("Gene Combinations")
ax.set_ylabel("SQ Values")
ax.set_title("SQ Values for Different Gene Combinations")
ax.set_xticks(x, labels, rotation=45, ha="right")
ax.legend(labels, bbox_to_anchor=(1.05, 1), loc="upper left")

# Show the plot
plt.tight_layout()
plt.show()


### Gradient descent MCA, with no repeat overexpression allowed

In [ ]:
poi = pd.read_csv(
    "../../assays/Proteomics/Proteomics_Proteins_of_interest.csv", delimiter=";"
).drop([0, 15], axis=0)

ms = get_parameterised_model()
pars_for_mca = [par for par in ms.get_parameter_names() if "ksynth" in par]
y0 = {}
for i in ms.compounds:
    y0[i] = 0.01
y0.update(get_enzyme_concentrations())
y0.update(
    {
        "3PGA": 0.1,
        "ATP": 0.1,
        "NADPH": 0.1,
        "Fdred": 0.1,
        "ThiDP": 10,
        "chlorophyll-phytol": CHLA / ms.get_parameter("cell_volume"),
        "Cell_density": 1e1,
        "carotenoids": CAROT / ms.get_parameter("cell_volume"),
        "SQ": SQUALENE / ms.get_parameter("cell_volume"),
    }
)
s = Simulator(ms)
s.initialise(y0)
t_ss, y_ss = s.simulate_to_steady_state()

y0_ss = y_ss[-1]
pars_for_mca = [par for par in ms.get_parameter_names() if "ksynth" in par]
frcs_one_ox = {}
mca_results_one_ox = {}
modified_parameters_one_ox = []  # To keep track of modified parameters in order

for runs in np.arange(0, 10):
    crc, frc = mca.get_response_coefficients_df(ms, pars_for_mca, y0_ss)
    s = Simulator(ms)
    s.initialise(y0_ss)
    t_ss, y_ss = s.simulate_to_steady_state()
    mca_results_one_ox[runs] = s.get_fluxes_df().iloc[-1]
    frcs_one_ox[runs] = frc.copy()
    y0_ss = y_ss[-1]

    # Filter out already modified parameters
    # frc_filtered = frc.copy()
    frc_filtered = frc.drop(index=modified_parameters_one_ox)
    most_positive = (frc_filtered["Sqs"] + frc_filtered["Biomass"]).idxmax()
    overexpression_amount = 1.331057 * poi[poi["Enzyme"] == most_positive[:-7]]["kDa"]
    ms.update_parameters(
        {
            most_positive: ms.get_parameter(most_positive)
            + overexpression_amount / (5.5 * 60)
        }
    )
    modified_parameters_one_ox.append(
        most_positive
    )  # Add the modified parameter to the list

# Now, modified_parameters contains the parameters in the order they were modified


In [ ]:
fig, ax = plt.subplots(2, figsize=(10, 10))
# Assuming frcs_one_ox and modified_parameters are defined as in your previous code
s_coll = pd.DataFrame()
Squalene_coll = []
for i, f in enumerate(frcs_one_ox):
    s_coll = pd.concat(
        [
            s_coll,
            (frcs_one_ox[f]["Sqs"] + abs(frcs_one_ox[f]["Biomass"].min())).rename(f),
        ],
        axis=1,
    )
    # s_coll = pd.concat([s_coll, (frcs_one_ox[f]["Biomass"]).rename(f)], axis=1)

    Squalene_coll.append(
        mca_results_one_ox[i]["Sqs"] / mca_results_one_ox[i]["Biomass"] * 410.73 / 1000
    )

print(s_coll)

# Create the heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(s_coll, annot=False, fmt=".2f", cmap="RdBu_r", center=0, ax=ax[0])

ax[1].plot(Squalene_coll)
ax[1].set_ylim(0)
# Draw rectangles around the overexpressed genes
for run, gene in enumerate(modified_parameters_one_ox):
    ax[0].add_patch(
        plt.Rectangle(
            (run, s_coll.index.get_loc(gene)),
            1,
            1,
            fill=False,
            edgecolor="red",
            linewidth=3,
        )
    )
ax[0].set_xlabel("No. of genes overexpressed")
ax[0].set_ylabel("Adjustable parameters")
ax[1].set_ylabel("Intracellular squalene concentration [$g\ L^{-1}$]")
ax[1].set_xlabel("No. of genes overexpressed")
ax[1].set_xticks(np.arange(0, 10, 1))
plt.show()

# SQ_coll.append(mca_results_one_ox[i]["Biomass"]*mca_results_one_ox[i]["Sqs"])

### Gradient descent MCA, repeat overexpression allowed

In [ ]:
poi = pd.read_csv(
    "../../assays/Proteomics/Proteomics_Proteins_of_interest.csv", delimiter=";"
).drop([0, 15], axis=0)

ms = get_parameterised_model()
pars_for_mca = [par for par in ms.get_parameter_names() if "ksynth" in par]
y0 = {}
for i in ms.compounds:
    y0[i] = 0.01
y0.update(get_enzyme_concentrations())
y0.update(
    {
        "3PGA": 0.1,
        "ATP": 0.1,
        "NADPH": 0.1,
        "Fdred": 0.1,
        "ThiDP": 10,
        "chlorophyll-phytol": CHLA / ms.get_parameter("cell_volume"),
        "Cell_density": 1e1,
        "carotenoids": CAROT / ms.get_parameter("cell_volume"),
        "SQ": SQUALENE / ms.get_parameter("cell_volume"),
    }
)
s = Simulator(ms)
s.initialise(y0)
t_ss, y_ss = s.simulate_to_steady_state()

y0_ss = y_ss[-1]
pars_for_mca = [par for par in ms.get_parameter_names() if "ksynth" in par]
frcs = {}
mca_results = {}
modified_parameters = []  # To keep track of modified parameters in order

for runs in np.arange(0, 10):
    crc, frc = mca.get_response_coefficients_df(ms, pars_for_mca, y0_ss)
    s = Simulator(ms)
    s.initialise(y0_ss)
    t_ss, y_ss = s.simulate_to_steady_state()
    mca_results[runs] = s.get_fluxes_df().iloc[-1]
    frcs[runs] = frc.copy()
    y0_ss = y_ss[-1]

    # Filter out already modified parameters
    frc_filtered = frc.copy()
    # frc_filtered = frc.drop(index=modified_parameters)
    most_positive = (frc_filtered["Sqs"] + frc_filtered["Biomass"]).idxmax()
    overexpression_amount = 1.331057 * poi[poi["Enzyme"] == most_positive[:-7]]["kDa"]
    ms.update_parameters(
        {
            most_positive: ms.get_parameter(most_positive)
            + overexpression_amount / (5.5 * 60)
        }
    )
    modified_parameters.append(most_positive)  # Add the modified parameter to the list

# Now, modified_parameters contains the parameters in the order they were modified


In [ ]:
fig, ax = plt.subplots(2, figsize=(10, 10))
# Assuming frcs and modified_parameters are defined as in your previous code
s_coll = pd.DataFrame()
Squalene_coll = []
for i, f in enumerate(frcs):
    s_coll = pd.concat(
        [s_coll, (frcs[f]["Sqs"] + abs(frcs[f]["Biomass"].min())).rename(f)], axis=1
    )
    # s_coll = pd.concat([s_coll, (frcs[f]["Sqs"]).rename(f)], axis=1)

    Squalene_coll.append(
        mca_results[i]["Sqs"] / mca_results[i]["Biomass"] * 410.73 / 1000
    )
print(s_coll)

# Create the heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(
    s_coll,
    annot=False,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    ax=ax[0],
    cbar_kws={"label": "Flux response coefficient"},
)

ax[1].plot(Squalene_coll)
ax[1].set_ylim(0)
# Draw rectangles around the overexpressed genes
for run, gene in enumerate(modified_parameters):
    ax[0].add_patch(
        plt.Rectangle(
            (run, s_coll.index.get_loc(gene)),
            1,
            1,
            fill=False,
            edgecolor="red",
            linewidth=3,
        )
    )

ax[0].set_xlabel("No. of genes overexpressed")
ax[0].set_ylabel("Adjustable parameters")
ax[1].set_ylabel("Intracellular squalene concentration [$g\ L^{-1}$]")
ax[1].set_xlabel("No. of genes overexpressed")
ax[1].set_xticks(np.arange(0, 10, 1))
plt.show()

# SQ_coll.append(mca_results[i]["Biomass"]*mca_results[i]["Sqs"])

In [ ]:
synths = [par for par in ms.get_parameters() if "ksynth" in par]
default = get_parameterised_model()
for par in synths:
    print(par, ms.get_parameters()[par] - default.get_parameters()[par])


In [ ]:
ms = get_parameterised_model()
y0 = {}
for i in ms.compounds:
    y0[i] = 0.01
y0.update(get_enzyme_concentrations())
y0.update(
    {
        "3PGA": 0.1,
        "ATP": 0.1,
        "NADPH": 0.1,
        "Fdred": 0.1,
        "ThiDP": 10,
        "chlorophyll-phytol": CHLA / ms.get_parameter("cell_volume"),
        "Cell_density": 1e1,
        "carotenoids": CAROT / ms.get_parameter("cell_volume"),
        "SQ": SQUALENE / ms.get_parameter("cell_volume"),
        "light": 100,
    }
)
s = Simulator(ms)
s.initialise(y0)
t_ss, y_ss = s.simulate_to_steady_state()
y0_ss = y_ss[-1]
pars_for_mca = [par for par in ms.get_parameter_names() if "ksynth" in par]
pars_for_mca.append("light")

crc, frc = mca.get_response_coefficients_df(ms, pars_for_mca, y0_ss)

In [ ]:
fig, ax = plt.subplots()
sns.heatmap(
    pd.DataFrame(
        [
            frc["ATP production"],
            frc["NADPH production"],
            frc["Fdred production"],
            frc["3PGA production"],
        ]
    ).transpose(),
    ax=ax,
    cmap="vlag",
    center=0,
    xticklabels=["ATP", "NADPH", "Fdred", "3PGA"],
    cbar_kws={"label": "Response coefficient"},
)


In [ ]:
fig, ax = plt.subplots()
sns.heatmap(
    pd.DataFrame(
        [frc["Sqs"], frc["chlorophyll_synthesis2"], frc["carotenoids"], frc["Biomass"]]
    ).transpose(),
    ax=ax,
    cmap="vlag",
    center=0,
    xticklabels=["Squalene", "Chlorophyll", "Carotenoids", "Biomass"],
    cbar_kws={"label": "Response coefficient"},
)


In [ ]:
ms = get_model(mode="blast", growth_mode="dynamic")

y0 = {}
for i in ms.compounds:
    y0[i] = 1
y0.update(get_enzyme_concentrations())
y0.update(
    {
        "3PGA": 0.1,
        "ATP": 0.1,
        "NADPH": 0.1,
        "Fdred": 0.1,
        "ThiDP": 0.0,
        "chlorophyll-phytol": 10,
        "Cell_density": 1e1,
        "carotenoids": 3,
    }
)
print(y0)
ms.update_parameters(
    {
        "ShcKcat": 0,
        "light": 100,
        "ATP_out_metabolism": 0.5,
        "3PGA_out_metabolism": 0.5,
        "NADPH_out_metabolism": 0.5,
        "Fdred_out_metabolism": 0.5,
        "out": 0,
        "Kna_DXS": 0.0001,
        "DXS_na_kdeg": 1 / 50000,
        "DXS_inh_str": 0.001,
        # "ATP_tot":430.143, "NADP_tot":26.805, "3PGA_max":1000, "Fd_tot":3.597,
        "cell_volume": 2e-15,
        "protein_content": 0.445,
    }
)
c_flux = []
conc_collection = []
flux_collection = []
for i in np.logspace(1, 1000, num=3):
    print(i)
    y0.update({"Cell_density": i})
    s = Simulator(ms)
    s.initialise(y0)
    t_steady, y_ss = s.simulate(10000, rtol=1e-4, atol=1e-4)
    print(list(zip(ms.compounds, y_ss[-1])))
    print([c for c in list(zip(ms.compounds, y_ss[-1])) if c[1] < 0])
    # s.plot_selection(m.compounds[:25])
    print(s.get_full_results_df())
    conc_collection.append(s.get_full_results_df())
    flux_collection.append(s.get_fluxes_df())

### Steady state simulation, no cell density

In [ ]:
y0.update(
    {
        "light": 100,
    }
)
pd.options.display.float_format = "{:.6f}".format
metabolite_concentrations = pd.read_csv(
    "Parameterisation_data/Metabolite_concentrations.csv", delimiter=";"
)
metabolite_concentrations = metabolite_concentrations.dropna(how="all")
metabolite_concentrations.set_index(metabolite_concentrations.columns[0], inplace=True)
ms = get_parameterised_model()

font = {"size": 22}
fig, ax = plt.subplots(figsize=(10, 5))
plt.rc("font", **font)
# plt.style.use("default")
s = Simulator(ms)
s.initialise(y0)
t_steady, y_steady = s.simulate_to_steady_state()
if y_steady is None:
    print("No Steady state was reached")
    s.initialise(y0)
    y_ss = s.simulate(604800)[-1]
else:
    y_ss = y_steady
    print("Steady state was reached")
y_ss = y_ss[-1].reshape(39)
metabolite_concentrations.loc["Squalene"]["Measured concentration"] = (
    metabolite_concentrations.loc["Squalene"]["Measured concentration"]
    + metabolite_concentrations.loc["Hopanoids"]["Measured concentration"]
)
metabolite_concentrations.loc["Hopanoids"]["Measured concentration"] = 0
metabolite_concentrations["Measured concentration"] = (
    metabolite_concentrations["Measured concentration"] / 1000
)
metabolite_concentrations["Standard deviation"] = (
    metabolite_concentrations["Standard deviation"] / 1000
)

metabolite_concentrations["Simulated concentration"] = y_ss[:23]
# metabolite_concentrations.drop(["CDP-MEP","ThiDP", "GPP", "FPP", "GGPP", "Phytol", "chlorophyll-GGPP"], inplace=True)
metabolite_concentrations.rename(
    index={"chlorophyll-phytol": "Chlorophyll", "carot": "Carotenoids"}, inplace=True
)

enzyme_comparison = pd.DataFrame.from_dict(
    get_enzyme_concentrations(), orient="index", columns=["Measured concentration"]
)
enzyme_comparison.rename(index={"DXS_nonactive": "DXS_M", "DXS": "DXS_D"}, inplace=True)
simulated_enzymes = pd.DataFrame(
    [i for i in y_ss], index=ms.compounds, columns=["Simulated concentration"]
)
simulated_enzymes.rename(index={"DXS_nonactive": "DXS_M", "DXS": "DXS_D"}, inplace=True)

enzyme_comparison["Simulated concentration"] = simulated_enzymes[
    "Simulated concentration"
]
standard_deviations = pd.DataFrame(
    columns=metabolite_concentrations.drop("Standard deviation", axis=1).columns,
    index=metabolite_concentrations.index,
)
standard_deviations["Measured concentration"] = metabolite_concentrations[
    "Standard deviation"
]
standard_deviations.replace(np.nan, 0, inplace=True)
target_concentrations = pd.Series(
    index=ms.compounds,
    data=pd.concat(
        [
            metabolite_concentrations["Measured concentration"],
            enzyme_comparison["Measured concentration"],
        ]
    ),
)

metabolite_concentrations.drop("Standard deviation", axis=1).plot.bar(
    ax=ax, width=0.8, logy=True, yerr=standard_deviations
)

# print(*[i for i in list(zip(m.compounds, y_ss))], sep="\n")
# bar = enzyme_comparison.plot.bar(ax=ax,logy=True, width=0.7).get_figure()

ax.set_ylim(0, 320)
ax.set_ylabel("Intracellular concentration [mM]")
ax.set_xlabel("Metabolite")
plt.xticks(rotation=90)
plt.subplots_adjust(left=0.1, right=0.9, top=1, bottom=0.01)
plt.show()

enzyme_comparison

In [ ]:
print(
    "Doubling time [h]:",
    (
        np.log(2)
        / np.log(1 + (s.get_full_results_df()["Growth_rate"].values.astype("float")))
    )
    / 3600,
)
print(
    "% growth rate per second",
    s.get_full_results_df()["Growth_rate"].values.astype("float"),
)
print(
    "% growth rate per hour",
    s.get_full_results_df()["Growth_rate"].values.astype("float") * 3600,
)

In [ ]:
print(
    "Doubling time [h]:",
    (
        np.log(2)
        / np.log(1 + (s.get_full_results_df()["Growth_rate"].values.astype("float")))
    )
    / 3600,
)
print(
    "% growth rate per second",
    s.get_full_results_df()["Growth_rate"].values.astype("float"),
)
print(
    "% growth rate per hour",
    s.get_full_results_df()["Growth_rate"].values.astype("float") * 3600,
)

In [ ]:
lin = np.linspace(1, 1, 1)
linsqs = np.linspace(1, 10, 9)
linIspH = np.linspace(1, 1, 1)
def_model = get_parameterised_model()
ms = get_parameterised_model()
results = pd.DataFrame(
    index=np.arange(0, len(lin) * len(linsqs) * len(linIspH), 1),
    columns=[
        "DXS_ksynth",
        "Sqs_ksynth",
        "IspH_ksynth",
        "Squalene",
        "Chlorophyll",
        "Carotenoids",
        "Growth_rate",
    ],
)
u = 0
for o, k in enumerate(def_model.get_parameter("Sqs_ksynth") * linsqs):
    ms.update_parameters({"Sqs_ksynth": k})
    for i, j in enumerate(def_model.get_parameter("DXS_ksynth") * lin):
        ms.update_parameters({"DXS_ksynth": j})
        for p, l in enumerate(def_model.get_parameter("IspH_ksynth") * linIspH):
            ms.update_parameters({"IspH_ksynth": l})
            s = Simulator(ms)
            s.initialise(y0)
            _, y_ss = s.simulate_to_steady_state()
            results.iloc[u]["DXS_ksynth"] = ms.get_parameter("DXS_ksynth")
            results.iloc[u]["Sqs_ksynth"] = ms.get_parameter("Sqs_ksynth")
            results.iloc[u]["IspH_ksynth"] = ms.get_parameter("IspH_ksynth")
            if y_ss is not None:
                results.iloc[u]["Squalene"] = y_ss[0][17]
                results.iloc[u]["Carotenoids"] = y_ss[0][18]

                results.iloc[u]["Chlorophyll"] = y_ss[0][20]
                results.iloc[u]["Growth_rate"] = (
                    s.get_full_results_df()["Growth_rate"].values.astype("float") * 3600
                )
            u += 1


In [ ]:
plt.style.use("default")
colors = ["#1f77b4", "green", "orange"]
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
results = results.astype("float64")
print(results)
bar = (
    results.dropna(how="all")
    .drop(columns=["DXS_ksynth", "Sqs_ksynth", "IspH_ksynth", "Growth_rate"])
    .plot(
        kind="bar",
        stacked=True,
        ax=ax,
        color=colors,
    )
)
bar.set_ylabel("Concentration [µM]")
line = results.dropna(how="all")["Growth_rate"].plot(
    kind="line", stacked=True, ax=ax, secondary_y=True, color="black"
)
line.set_ylabel("Growth rate [$h^-1$]")
plt.legend(["Growth rate"], loc="upper left")

# bar2 = results.sort_values(by="Squalene")["Growth_rate"].plot(kind="bar",ax=ax[1])
# bar.set_xticklabels(lin+linsqs, rotation=0)
# bar2.set_xticklabels(lin, rotation=0)
xlabel = []
for i in linsqs:
    for o in lin:
        for p in linIspH:
            xlabel.append(str(int(p)))
bar.set_xticklabels(xlabel, rotation=0)
# bar2.set_xticklabels(xlabel, rotation=0)
# bar2.set_ylabel("Growth rate [$h^{-1}$]")
# bar2.set_xlabel("fold change in Dxs synthesis")
bar.set_xlabel("fold change in Dxs synthesis")

# plt.savefig("Dxs_10x_ox.png")

In [ ]:
CELL_VOLUME = 0.6e-15
enzymes = get_enzyme_concentrations(cell_volume=CELL_VOLUME)
enzymes.pop("ThiG")
enzymes.pop("Shc")
enzymes.pop("DXS_nonactive")


def simplified_MM(Et, kcat, S):
    return (Et * kcat * S) / S


def MM(Et, kcat, S, Km):
    return (Et * kcat * S) / (S + Km)


Chla_conc = (
    (CHLA * 1000 * 4) + (CAROT * 1000 * 8) + (SQUALENE * 1000 * 6)
) / CELL_VOLUME  # Overall production of DXP units

doubling_time = np.log(2) / 0.13 * 60  # Approx. maximum growth rate in turbidostat
Chla_production = Chla_conc / doubling_time  # Unit: µM/min
# Calculate kcat for each enzyme
MIN_KCAT_VALUES = {
    enzyme + "Kcat": Chla_production / Et for enzyme, Et in enzymes.items()
}
MIN_KCAT_VALUES["ThiGKcat"] = 1000
MIN_KCAT_VALUES["SqsKcat"] = SQUALENE * 1000 / CELL_VOLUME / enzymes["Sqs"]
MIN_KCAT_VALUES["ChlPKcat"] = CHLA * 1000 / CELL_VOLUME / enzymes["ChlP"]
MIN_KCAT_VALUES["ChlGKcat"] = CHLA * 1000 / CELL_VOLUME / enzymes["ChlG"]
MIN_KCAT_VALUES["CrtBKcat"] = CAROT * 1000 / CELL_VOLUME / enzymes["CrtB"]
MIN_KCAT_VALUES["IdiKcat"] = 100
MIN_KCAT_VALUES["IspDKcat"] = 1000
MIN_KCAT_VALUES["IspH1Kcat"] = MIN_KCAT_VALUES["IspHKcat"]
MIN_KCAT_VALUES["IspH2Kcat"] = MIN_KCAT_VALUES["IspHKcat"]

MIN_KCAT_VALUES["CrtE1Kcat"] = MIN_KCAT_VALUES["CrtEKcat"]
MIN_KCAT_VALUES["CrtE2Kcat"] = MIN_KCAT_VALUES["CrtEKcat"]
MIN_KCAT_VALUES["CrtE3Kcat"] = MIN_KCAT_VALUES["CrtEKcat"]

ms = get_model(mode="blast", growth_mode="steady", cell_volume=CELL_VOLUME)
ms.update_parameters(
    {
        "ShcKcat": 0,
        "Shc_ksynth": 0,
        "light": 400,
        "ATP_out_metabolism": 0.5,
        "3PGA_out_metabolism": 0.5,
        "NADPH_out_metabolism": 0.5,
        "Fdred_out_metabolism": 0.5,
        "out": 0,  # .0003137085008420509,
        # "ATP_tot":430.143, "NADP_tot":26.805, "3PGA_max":1000, "Fd_tot":3.597,
        "cell_volume": CELL_VOLUME,
        "protein_content": 0.445,
        "DXS_inh_str": 0.001,
    }
)

def_model = ms.copy()

In [ ]:
ms.compounds[16]

In [ ]:
lin = np.linspace(0, 1, 2)
linsqs = np.linspace(0, 1, 2)
linIspH = np.linspace(0, 1, 2)
linCrtE = np.linspace(1, 0.5, 2)
with ms:
    results = pd.DataFrame(
        index=np.arange(0, len(lin) * len(linsqs) * len(linIspH) * len(linCrtE), 1),
        columns=["Squalene", "Chlorophyll", "Carotenoids", "Growth_rate"],
    )
    u = 0
    for o, k in enumerate(def_model.get_parameter("Sqs_ksynth") + (linsqs * 0.0001)):
        ms.update_parameters({"Sqs_ksynth": k})
        for i, j in enumerate(def_model.get_parameter("DXS_ksynth") + (lin * 0.0001)):
            ms.update_parameters({"DXS_ksynth": j})
            for p, l in enumerate(
                def_model.get_parameter("IspH_ksynth") + (linIspH * 0.0001)
            ):
                ms.update_parameters({"IspH_ksynth": l})
                for w, q in enumerate(def_model.get_parameter("CrtE_ksynth") * linCrtE):
                    ms.update_parameters({"CrtE_ksynth": q})
                    s = Simulator(ms)
                    s.initialise(y0)
                    _, y_ss = s.simulate_to_steady_state()
                    if y_ss is not None:
                        results.iloc[u]["Squalene"] = y_ss[0][16]
                        results.iloc[u]["Chlorophyll"] = y_ss[0][21]
                        results.iloc[u]["Carotenoids"] = y_ss[0][17]
                        results.iloc[u]["Growth_rate"] = (
                            s.get_full_results_df()["Growth_rate"].values.astype(
                                "float"
                            )
                            * 60
                        )
                    u += 1
    print(results)
results = results.astype("float64")
results.loc[-1] = pd.Series([math.nan, math.nan, math.nan])
results.index = results.index + 1
results.sort_index(inplace=True)
print(results)

In [ ]:
colors = ["#1f77b4", "green", "orange"]
plt.style.use("default")
fig, ax = plt.subplots(1, 1, figsize=(15, 5))
plt.gcf().subplots_adjust(bottom=0.17)
# bar = results.drop(["Growth_rate"],axis=1).plot(kind="bar", stacked=True, ax=ax, color=colors)
print(results)
xlabel = []
xlabel.append("Sqs\nDxs\nIspG\nCrtE")
for i in linsqs:
    for o in lin:
        for p in linIspH:
            for w in linCrtE:
                xlabel.append(
                    str(
                        str(int(i)) + "\n" + str(int(o)) + "\n" + str(p) + "\n" + str(w)
                    )
                )
xlabeldf = pd.Series(xlabel, index=results.index)
xlabeldf.loc[0] = "Sqs\nDxs\nIspH\nCrtE"
print(xlabeldf)
results_constrained = pd.DataFrame(results)
# results_constrained.dropna(how="all",inplace=True)
# results_constrained.loc[-1] = pd.Series([math.nan, math.nan, math.nan])
results_constrained.index = results_constrained.index + 1
results_constrained.sort_index(inplace=True)
results_constrained["Squalene"] = results_constrained["Squalene"]
results_constrained["Chlorophyll"] = results_constrained["Chlorophyll"]
results_constrained["Carotenoids"] = results_constrained["Carotenoids"]
# results_constrained.dropna(how="all", inplace=True)
print(results_constrained)
productivity = pd.Series(index=results_constrained.index)
productivity.name = "Squalene production over 7 d"
for i in results_constrained.index:
    productivity.loc[i] = (
        results_constrained.loc[i]["Squalene"]
        * (
            10**10
            * (4 * 10**-15)
            * ((1 + results_constrained.loc[i]["Growth_rate"]) ** (7 * 24))
        )
        * 410.73
    )
results_constrained.rename(
    {
        "Squalene": "Intracellular squalene concentration",
        "Chlorophyll": "Intracellular chlorophyll concentration",
        "Carotenoids": "Intracellular carotenoid concentration",
    },
    inplace=True,
)

productivity = productivity.reset_index()
productivity.drop(columns="index", inplace=True)
productivity.iloc[0] = results_constrained.iloc[0][0]
line = productivity.plot(
    kind="line",
    color="black",
    ax=ax,
    secondary_y=True,
    legend="Squalene production in 7 d",
)
line.set_xticks(np.arange(0, len(xlabeldf), 1))
# line.set_xticklabels(xlabeldf.values)
line.set_ylabel("Production after 7 d [mg/L]")
bar = results_constrained.drop(["Growth_rate"], axis=1).plot(
    kind="bar", stacked=True, ax=ax, color=colors
)

bar.legend(
    [
        "Intracellular squalene concentration",
        "Intracellular chlorophyll concentration",
        "Intracellular carotenoid concentration",
    ]
)
# bar.set_xticklabels(xlabeldf.values, rotation=0)
# bar.set_xticklabels(xlabel, rotation=0)
plt.tick_params(bottom=False)
bar.set_xlabel("fold change in synthesis rates")
bar.set_ylabel("Concentration [mM]")
# plt.savefig("figures/Overexpressions_productivity.png")# 0.0195 0.0156 80% growth rate

In [ ]:
y0_ss = {}
starting_conc_ss_half = {}

print(ms.compounds)
ss_metabolites = {}

half_metabolites = {}
for i, d in enumerate(y0):
    ss_metabolites[d] = y_ss[i]
    half_metabolites[d] = y_ss[i] / 2
    # print(len(starting_conc))

y0_ss.update(ss_metabolites)
starting_conc_ss_half.update(half_metabolites)

print(y0_ss)


In [ ]:
pars_to_change = [parameter for parameter in ms.parameters if "ksynth" in parameter]
pars_to_change.remove("Shc_ksynth")
pars_to_change.append("light")
ms = get_model(mode="blast")
print(y0)
ms.update_parameters(
    {
        "Shc_ksynth": 0,
        "out": 7e-07,
        "Kna_DXS": 0.00001,
        "DXS_na_kdeg": 1 / 50000,
        "DXS_inh_str": 0.005,
        "light": 100,
    }
)

crc, frc = mca.get_response_coefficients_df(ms, pars_to_change, y0)

In [ ]:
crc2 = crc.dropna(how="all")
frc2 = frc.dropna(how="all")


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
sns.set(font_scale=1)
print(frc2.index)
frc2["Chla_synthesis"] = frc2["chlorophyll_synthesis1"]
# sns.heatmap(crc2["SQ"].sort_values(ascending=False).iloc[:].values.reshape(len(crc2["SQ"].sort_values(ascending=False).iloc[:]),1), center = 0, cmap="vlag", ax=ax,yticklabels=crc2["SQ"].sort_values(ascending=False).index[:], xticklabels=[])
# #sns.heatmap(frc2["SQ"].sort_values(ascending=False).iloc[:].values.reshape(len(frc2["SQ"].sort_values(ascending=False).iloc[:]),1), center = 0, cmap="vlag", ax=ax[1],yticklabels=frc2["SQ"].sort_values(ascending=False).index[:], xticklabels=[])
# sns.heatmap(crc2["carot"].sort_values(ascending=False).iloc[:].values.reshape(len(crc2["carot"].sort_values(ascending=False).iloc[:]),1), center = 0, cmap="vlag", ax=ax,yticklabels=crc2["carot"].sort_values(ascending=False).index[:], xticklabels=[])
# #sns.heatmap(frc2["carotenoids_out"].sort_values(ascending=False).iloc[:].values.reshape(len(frc2["carotenoids_out"].sort_values(ascending=False).iloc[:]),1), center = 0, cmap="vlag", ax=ax,yticklabels=frc2["carotenoids_out"].sort_values(ascending=False).index[:], xticklabels=[])
# sns.heatmap(crc2["chlorophyll-phytol"].sort_values(ascending=False).iloc[:].values.reshape(len(crc2["chlorophyll-phytol"].sort_values(ascending=False).iloc[:]),1), center = 0, cmap="vlag", ax=ax,yticklabels=crc2["chlorophyll-phytol"].sort_values(ascending=False).index[:], xticklabels=[])
# sns.heatmap(frc2["chlorophyll_out"].sort_values(ascending=False).iloc[:].values.reshape(len(frc2["chlorophyll_out"].sort_values(ascending=False).iloc[:]),1), center = 0, cmap="vlag", ax=ax,yticklabels=frc2["chlorophyll_out"].sort_values(ascending=False).index[:], xticklabels=[])
sns.heatmap(
    pd.DataFrame(
        [frc2["sqs1"], frc2["Chla_synthesis"], frc2["carotenoids"]]
    ).transpose(),
    ax=ax,
    cmap="vlag_r",
    center=0,
    xticklabels=["Squalene", "Chlorophyll", "Carotenoids"],
    cbar_kws={"label": "Response coefficient"},
)
# sns.heatmap(pd.DataFrame([crc2["SQ"], crc2["chlorophyll-phytol"],crc2["carot"]]).transpose(), ax=ax, cmap="vlag_r",center=0, xticklabels=["Squalene", "Chlorophyll", "Carotenoids"],cbar_kws={'label': 'Response coefficient'})
ax.set_ylabel("Changed flux", fontsize=14)
# ax.set_xlabel("Metabolite", loc="center",fontsize=14)
ax.set_xticklabels(["Squalene", "Chlorophyll", "Carotenoids"], fontsize=14)
# fig.suptitle("Increased squalene production upon induction of gene of interest")
plt.tight_layout()
fig.suptitle("Flux response coefficients", y=1.05, fontsize=18, x=0.5)
plt.savefig("figures/20240502MCA_Flux_coefficients")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
sns.set(font_scale=1)
print(frc2.index)
frc2["Chla_synthesis"] = frc2["chlorophyll_synthesis1"]
sns.heatmap(
    pd.DataFrame([crc2["SQ"], crc2["chlorophyll-phytol"], crc2["carot"]]).transpose(),
    ax=ax,
    cmap="vlag_r",
    center=0,
    xticklabels=["Squalene", "Chlorophyll", "Carotenoids"],
    cbar_kws={"label": "Response coefficient"},
)
# #sns.heatmap(frc2["SQ"].sort_values(ascending=False).iloc[:].values.reshape(len(frc2["SQ"].sort_values(ascending=False).iloc[:]),1), center = 0, cmap="vlag", ax=ax[1],yticklabels=frc2["SQ"].sort_values(ascending=False).index[:], xticklabels=[])
# sns.heatmap(crc2["carot"].sort_values(ascending=False).iloc[:].values.reshape(len(crc2["carot"].sort_values(ascending=False).iloc[:]),1), center = 0, cmap="vlag", ax=ax,yticklabels=crc2["carot"].sort_values(ascending=False).index[:], xticklabels=[])
# #sns.heatmap(frc2["carotenoids_out"].sort_values(ascending=False).iloc[:].values.reshape(len(frc2["carotenoids_out"].sort_values(ascending=False).iloc[:]),1), center = 0, cmap="vlag", ax=ax,yticklabels=frc2["carotenoids_out"].sort_values(ascending=False).index[:], xticklabels=[])
# sns.heatmap(crc2["chlorophyll-phytol"].sort_values(ascending=False).iloc[:].values.reshape(len(crc2["chlorophyll-phytol"].sort_values(ascending=False).iloc[:]),1), center = 0, cmap="vlag", ax=ax,yticklabels=crc2["chlorophyll-phytol"].sort_values(ascending=False).index[:], xticklabels=[])
# sns.heatmap(frc2["chlorophyll_out"].sort_values(ascending=False).iloc[:].values.reshape(len(frc2["chlorophyll_out"].sort_values(ascending=False).iloc[:]),1), center = 0, cmap="vlag", ax=ax,yticklabels=frc2["chlorophyll_out"].sort_values(ascending=False).index[:], xticklabels=[])
# sns.heatmap(pd.DataFrame([frc2["sqs1"], frc2["Chla_synthesis"],frc2["carotenoids"]]).transpose(), ax=ax, cmap="vlag_r",center=0, xticklabels=["Squalene", "Chlorophyll", "Carotenoids"],cbar_kws={'label': 'Response coefficient'})
# sns.heatmap(pd.DataFrame([crc2["SQ"], crc2["chlorophyll-phytol"],crc2["carot"]]).transpose(), ax=ax, cmap="vlag_r",center=0, xticklabels=["Squalene", "Chlorophyll", "Carotenoids"],cbar_kws={'label': 'Response coefficient'})
ax.set_ylabel("Changed flux", fontsize=14)
# ax.set_xlabel("Metabolite", loc="center",fontsize=14)
ax.set_xticklabels(["Squalene", "Chlorophyll", "Carotenoids"], fontsize=14)
# fig.suptitle("Increased squalene production upon induction of gene of interest")
plt.tight_layout()
fig.suptitle("Flux response coefficients", y=1.05, fontsize=18, x=0.5)
plt.savefig("figures/20240502MCA_Conc_coefficients")

## Additional
